# MLP desde Cero en CPU con Validación (80/20)
### Implementación manual usando NumPy y PyTorch Tensors (sin Autograd ni nn.Module)

**Asignatura:** Tópicos en Inteligencia Artificial (UNSA)  
**Basado en:** *Introduction to Neural Network - Part I* por el M.Sc. Percy Maldonado Quispe  
**Fecha:** Mayo 2026  

---

### Características de esta Versión:
1. **Sin descargas automáticas**: Carga el dataset MNIST asumiendo que los archivos ya están descargados localmente en `./data` (`download=False`).
2. **Partición de Validación (80/20)**: El conjunto original de entrenamiento (60,000 imágenes) se divide aleatoriamente en un **80% para entrenar** (48,000 imágenes) y un **20% para validar** (12,000 imágenes).
3. **Puro CPU**: Todo el entrenamiento y evaluación se ejecutan exclusivamente en la CPU, garantizando portabilidad y bajo consumo.
4. **Implementación "a mano" (From Scratch)**: Propagación hacia adelante, cálculo del error, propagación hacia atrás y descenso de gradiente implementados manualmente usando tanto **NumPy** como **PyTorch Tensors** (sin `autograd`, `nn.Module` ni `torch.optim`).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import torch
import torchvision

# Configurar semillas aleatorias para reproductibilidad
np.random.seed(42)
torch.manual_seed(42)

print(f"Versión de PyTorch: {torch.__version__}")


## 1. Carga y Partición del Dataset MNIST (80/20)

Cargamos los datos asumiendo que ya se encuentran descargados en `./data` (`download=False`).  
Dividimos las 60,000 imágenes de entrenamiento en:
*   **80% Entrenamiento**: 48,000 imágenes.
*   **20% Validación**: 12,000 imágenes.


In [ ]:
# Cargar datos locales de MNIST (download=False)
raw_train = torchvision.datasets.MNIST("./data", train=True, download=False)
raw_test = torchvision.datasets.MNIST("./data", train=False, download=False)

# Convertir a matrices de NumPy y normalizar a [0, 1]
X_train_full = raw_train.data.numpy().reshape(-1, 784) / 255.0
y_train_full = raw_train.targets.numpy()

X_test = raw_test.data.numpy().reshape(-1, 784) / 255.0
y_test = raw_test.targets.numpy()

# Partición 80/20 manual
m = len(X_train_full)
idx = np.random.permutation(m)
split_limit = int(0.8 * m)

train_idx = idx[:split_limit]
val_idx = idx[split_limit:]

X_train = X_train_full[train_idx]
y_train = y_train_full[train_idx]

X_val = X_train_full[val_idx]
y_val = y_train_full[val_idx]

print(f"Entrenamiento (80%): {X_train.shape} | Etiquetas: {y_train.shape}")
print(f"Validación    (20%): {X_val.shape}  | Etiquetas: {y_val.shape}")
print(f"Prueba             : {X_test.shape}  | Etiquetas: {y_test.shape}")


## Parte 1 -- MLP desde cero con NumPy (CPU)

Implementación pura en NumPy utilizando funciones de activación Sigmoide (ocultas) y Softmax (salida), con entropía cruzada como función de coste y descenso de gradiente estándar.


In [ ]:
class MLP_NumPy:
    """MLP con forward/backward explicito en NumPy. Sin frameworks."""

    def __init__(self, dims):
        # Xavier init: escala por sqrt(1/fan_in)
        self.W = [np.random.randn(dims[i], dims[i+1]) * np.sqrt(1/dims[i])
                  for i in range(len(dims)-1)]
        self.b = [np.zeros((1, dims[i+1]))
                  for i in range(len(dims)-1)]

    @staticmethod
    def sigmoid(z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    @staticmethod
    def softmax(z):
        e = np.exp(z - z.max(axis=1, keepdims=True))
        return e / e.sum(axis=1, keepdims=True)

    def forward(self, X):
        self.a = [X]
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            z = self.a[-1] @ W + b
            is_last = (i == len(self.W) - 1)
            self.a.append(self.softmax(z) if is_last else self.sigmoid(z))
        return self.a[-1]

    def cross_entropy(self, y_pred, y_true):
        Y = np.eye(10)[y_true]
        return -np.mean(np.sum(Y * np.log(y_pred + 1e-8), axis=1))

    def backward(self, y_true, lr):
        m     = len(y_true)
        Y     = np.eye(10)[y_true]
        delta = self.a[-1] - Y
        for i in reversed(range(len(self.W))):
            dW = self.a[i].T @ delta / m
            db = delta.mean(axis=0, keepdims=True)
            if i > 0:
                da    = delta @ self.W[i].T
                delta = da * self.a[i] * (1 - self.a[i])
            self.W[i] -= lr * dW
            self.b[i] -= lr * db

    def fit(self, X, y, epochs=20, batch_size=256, lr=0.1, X_val=None, y_val=None):
        hist = {"loss": [], "acc": [], "val_acc": []}
        m    = len(X)
        for epoch in range(1, epochs+1):
            idx = np.random.permutation(m)
            batch_losses = []
            for start in range(0, m, batch_size):
                sl    = idx[start:start+batch_size]
                y_hat = self.forward(X[sl])
                batch_losses.append(self.cross_entropy(y_hat, y[sl]))
                self.backward(y[sl], lr)
            loss = np.mean(batch_losses)
            acc  = self.accuracy(X, y)
            hist["loss"].append(loss)
            hist["acc"].append(acc)
            if X_val is not None:
                va = self.accuracy(X_val, y_val)
                hist["val_acc"].append(va)
                print(f"Epoca {epoch:2d} | loss={loss:.4f} | acc={acc:.4f} | val_acc={va:.4f}")
            else:
                print(f"Epoca {epoch:2d} | loss={loss:.4f} | acc={acc:.4f}")
        return hist

    def accuracy(self, X, y):
        return np.mean(np.argmax(self.forward(X), axis=1) == y)


In [ ]:
np.random.seed(42)
mlp_np = MLP_NumPy([784, 256, 128, 10])

t0 = time.time()
hist_np = mlp_np.fit(
    X_train, y_train,
    epochs=20, batch_size=256, lr=0.1,
    X_val=X_val, y_val=y_val
)
t_numpy = time.time() - t0
print(f"Tiempo NumPy (CPU): {t_numpy:.1f}s")


## Parte 2 -- MLP desde cero con PyTorch Tensors (CPU, sin Autograd)

Implementación homóloga a la anterior pero utilizando **Tensores de PyTorch** ejecutados de forma explícita en **CPU**. Los gradientes y actualizaciones se calculan a mano (con `requires_grad=False`), demostrando cómo se puede usar PyTorch puramente como una librería de álgebra lineal acelerada.


In [ ]:
class MLP_Tensor_Scratch:
    """MLP con forward/backward explicito usando Tensores de PyTorch en CPU.
    Sin autograd, sin nn.Module, sin optim."""

    def __init__(self, dims):
        self.device = torch.device("cpu")
        # Xavier init
        self.W = [torch.randn(dims[i], dims[i+1], device=self.device) * np.sqrt(1/dims[i])
                  for i in range(len(dims)-1)]
        self.b = [torch.zeros((1, dims[i+1]), device=self.device)
                  for i in range(len(dims)-1)]

    @staticmethod
    def sigmoid(z):
        return 1.0 / (1.0 + torch.exp(-torch.clamp(z, min=-500, max=500)))

    @staticmethod
    def softmax(z):
        max_z = torch.max(z, dim=1, keepdim=True).values
        e = torch.exp(z - max_z)
        return e / torch.sum(e, dim=1, keepdim=True)

    def forward(self, X):
        if not isinstance(X, torch.Tensor):
            X = torch.tensor(X, dtype=torch.float32, device=self.device)
        else:
            X = X.to(self.device)
            
        self.a = [X]
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            z = torch.matmul(self.a[-1], W) + b
            is_last = (i == len(self.W) - 1)
            self.a.append(self.softmax(z) if is_last else self.sigmoid(z))
        return self.a[-1]

    def cross_entropy(self, y_pred, y_true):
        m = y_true.shape[0]
        Y = torch.zeros(m, 10, device=self.device)
        Y[torch.arange(m), y_true] = 1.0
        return -torch.mean(torch.sum(Y * torch.log(y_pred + 1e-8), dim=1)).item()

    def backward(self, y_true, lr):
        m = y_true.shape[0]
        Y = torch.zeros(m, 10, device=self.device)
        Y[torch.arange(m), y_true] = 1.0
        
        delta = self.a[-1] - Y
        for i in reversed(range(len(self.W))):
            dW = torch.matmul(self.a[i].t(), delta) / m
            db = torch.mean(delta, dim=0, keepdim=True)
            if i > 0:
                da = torch.matmul(delta, self.W[i].t())
                delta = da * self.a[i] * (1.0 - self.a[i])
            self.W[i] -= lr * dW
            self.b[i] -= lr * db

    def fit(self, X, y, epochs=20, batch_size=256, lr=0.1, X_val=None, y_val=None):
        X_t = torch.tensor(X, dtype=torch.float32, device=self.device) if not isinstance(X, torch.Tensor) else X.to(self.device)
        y_t = torch.tensor(y, dtype=torch.long, device=self.device) if not isinstance(y, torch.Tensor) else y.to(self.device)
        
        if X_val is not None:
            X_val_t = torch.tensor(X_val, dtype=torch.float32, device=self.device) if not isinstance(X_val, torch.Tensor) else X_val.to(self.device)
            y_val_t = torch.tensor(y_val, dtype=torch.long, device=self.device) if not isinstance(y_val, torch.Tensor) else y_val.to(self.device)
        else:
            X_val_t, y_val_t = None, None
            
        hist = {"loss": [], "acc": [], "val_acc": []}
        m = X_t.shape[0]
        
        for epoch in range(1, epochs+1):
            idx = torch.randperm(m, device=self.device)
            batch_losses = []
            for start in range(0, m, batch_size):
                sl = idx[start:start+batch_size]
                y_hat = self.forward(X_t[sl])
                batch_losses.append(self.cross_entropy(y_hat, y_t[sl]))
                self.backward(y_t[sl], lr)
            loss = np.mean(batch_losses)
            acc = self.accuracy(X_t, y_t)
            hist["loss"].append(loss)
            hist["acc"].append(acc)
            if X_val_t is not None:
                va = self.accuracy(X_val_t, y_val_t)
                hist["val_acc"].append(va)
                print(f"Epoca {epoch:2d} | loss={loss:.4f} | acc={acc:.4f} | val_acc={va:.4f}")
            else:
                print(f"Epoca {epoch:2d} | loss={loss:.4f} | acc={acc:.4f}")
        return hist

    def accuracy(self, X, y):
        if not isinstance(X, torch.Tensor):
            X = torch.tensor(X, dtype=torch.float32, device=self.device)
        if not isinstance(y, torch.Tensor):
            y = torch.tensor(y, dtype=torch.long, device=self.device)
        y_hat = self.forward(X)
        predicciones = torch.argmax(y_hat, dim=1)
        return torch.mean((predicciones == y).float()).item()


In [ ]:
torch.manual_seed(42)
mlp_tensor = MLP_Tensor_Scratch([784, 256, 128, 10])

t0 = time.time()
hist_tensor = mlp_tensor.fit(
    X_train, y_train,
    epochs=20, batch_size=256, lr=0.1,
    X_val=X_val, y_val=y_val
)
t_tensor = time.time() - t0
print(f"Tiempo Tensor (CPU): {t_tensor:.1f}s")


## Parte 3 -- Comparación final y Visualización

Comparamos el rendimiento y velocidad de ambas implementaciones y visualizamos las curvas de pérdida y precisión.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric, title in zip(axes, ["loss", "val_acc"], ["Pérdida", "Accuracy (Validación)"]):
    ax.plot(hist_np[metric],  "b-o", markersize=4, label="NumPy (CPU)")
    ax.plot(hist_tensor[metric], "g-s", markersize=4, label="PyTorch Tensor (CPU)")
    ax.set(title=title, xlabel="Epoca"); ax.grid(alpha=.3); ax.legend()
plt.suptitle("NumPy vs PyTorch Tensor (Ambos en CPU entrenados desde Cero)", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

# Evaluar precisión en el conjunto de prueba final (10,000 imágenes)
acc_test_np = mlp_np.accuracy(X_test, y_test)
acc_test_tensor = mlp_tensor.accuracy(X_test, y_test)

sep = "=" * 75
print(sep)
print(f"  NumPy         (CPU) -> Acc Test = {acc_test_np*100:.2f}% | Tiempo Total = {t_numpy:.1f}s")
print(f"  PyTorch Tensor(CPU) -> Acc Test = {acc_test_tensor*100:.2f}% | Tiempo Total = {t_tensor:.1f}s")
print(sep)


In [ ]:
y_pred_np = np.argmax(mlp_np.forward(X_test), axis=1)
errores   = np.where(y_pred_np != y_test)[0]

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, idx in zip(axes.flat, errores[:16]):
    ax.imshow(X_test[idx].reshape(28, 28), cmap="gray")
    ax.set_title(f"R:{y_test[idx]} P:{y_pred_np[idx]}", fontsize=8)
    ax.axis("off")
plt.suptitle("Ejemplos mal clasificados (NumPy MLP)", fontsize=12)
plt.tight_layout(); plt.show()
